# Extract frozen GLIM vectors and validation controls
Use a GPU. Attach private derived dataset `thestonedape/task-aware-eegtotext` and private checkpoint dataset `thestonedape/glim-zuco-checkpoint`; enable Internet and private secret `GITHUB_TOKEN`. The notebook runs a five-condition one-row smoke before full extraction. Correct global vectors cover train and validation; matched-wrong, zero, and train-scale-matched Gaussian controls cover validation only. Test is inaccessible. Completed chunks are hash-validated and reused. To resume across Kaggle sessions, attach a previously saved partial output dataset containing the `vectors/` directory.

In [ ]:
REPO_URL = 'https://github.com/thestonedape/task-aware-eeg2text.git'
COMMIT = '0269ad60667737fd1c6d16b8b1a54f752cbdae3e'
WORKTREE = '/kaggle/working/SemKey'
GLIM_REPO_URL = 'https://github.com/justin-xzliu/GLIM.git'
GLIM_COMMIT = 'e1f202cb793cfe7292fbc0072a4c26a7dd0660d9'
GLIM_WORKTREE = '/kaggle/working/GLIM'
EXPECTED_INDEX_SHA256 = 'bdaaaf5c91d3c9eec16a0727825da996fd2186867245951bfdfdc92aab7738b0'
CHECKPOINT_SHA256 = '25fcd31d1d6cafc9a0656c50a4916ba6ee106884b269d347284784cc0522c8ba'
DONOR_SHA256 = '2f7cd3e7ba9713819bdc6ed90d18077997df395b9d05238d63e871d5f58ce75b'
SMOKE_OUTPUT = '/kaggle/working/frozen-glim-vector-smoke'
OUTPUT = '/kaggle/working/task-aware-eeg2text-frozen-glim-vectors'
BATCH_SIZE = 8
CHUNK_SIZE = 128
FORCE_FRESH = False
assert len(COMMIT) == len(GLIM_COMMIT) == 40
assert all(len(value) == 64 for value in (EXPECTED_INDEX_SHA256, CHECKPOINT_SHA256, DONOR_SHA256))

In [ ]:
import csv, glob, hashlib, json, os, platform, shutil, subprocess, sys, torch
from kaggle_secrets import UserSecretsClient
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator'
print({'python': platform.python_version(), 'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0)})
github_token = UserSecretsClient().get_secret('GITHUB_TOKEN')
assert github_token, 'Enable the private Kaggle Secret named GITHUB_TOKEN'
askpass = '/kaggle/working/git_askpass.py'
with open(askpass, 'w', encoding='utf-8') as handle:
    handle.write("#!/usr/bin/env python3\nimport os, sys\nprompt = sys.argv[1] if len(sys.argv) > 1 else ''\nprint(os.environ['GITHUB_TOKEN'] if 'Password' in prompt else 'x-access-token')\n")
os.chmod(askpass, 0o700)
clone_env = os.environ.copy()
clone_env.update({'GIT_ASKPASS': askpass, 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': github_token})
for path in (WORKTREE, GLIM_WORKTREE):
    if os.path.exists(path):
        shutil.rmtree(path)
try:
    subprocess.run(['git', 'clone', REPO_URL, WORKTREE], check=True, env=clone_env)
finally:
    os.remove(askpass)
    del github_token, clone_env
subprocess.run(['git', '-C', WORKTREE, 'checkout', '--detach', COMMIT], check=True)
assert subprocess.check_output(['git', '-C', WORKTREE, 'rev-parse', 'HEAD'], text=True).strip() == COMMIT
subprocess.run(['git', 'clone', GLIM_REPO_URL, GLIM_WORKTREE], check=True)
subprocess.run(['git', '-C', GLIM_WORKTREE, 'checkout', '--detach', GLIM_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', GLIM_WORKTREE, 'rev-parse', 'HEAD'], text=True).strip() == GLIM_COMMIT
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'lightning==2.4.0', 'torchmetrics==1.3.1', 'einops==0.8.0', 'timm==0.9.16', 'transformers==4.52.0'], check=True)
subprocess.run([sys.executable, os.path.join(WORKTREE, 'evaluation', 'test_protocol_manifests.py')], check=True)
subprocess.run([sys.executable, os.path.join(WORKTREE, 'evaluation', 'test_recoverability_protocol.py')], check=True)
subprocess.run([sys.executable, os.path.join(WORKTREE, 'evaluation', 'test_frozen_vector_contract.py')], check=True)

In [ ]:
manifest_paths = glob.glob('/kaggle/input/**/metadata/shard_manifest.json', recursive=True)
assert len(manifest_paths) == 1, ('Attach exactly one canonical sharded dataset', manifest_paths)
dataset_root = os.path.dirname(os.path.dirname(manifest_paths[0]))
checkpoint_paths = glob.glob('/kaggle/input/**/*.ckpt', recursive=True)
assert len(checkpoint_paths) == 1, ('Attach exactly one GLIM checkpoint', checkpoint_paths)
checkpoint = checkpoint_paths[0]
state = hashlib.sha256()
with open(checkpoint, 'rb') as handle:
    for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
        state.update(block)
assert state.hexdigest() == CHECKPOINT_SHA256
if FORCE_FRESH and os.path.exists(OUTPUT):
    shutil.rmtree(OUTPUT)
resume_markers = glob.glob('/kaggle/input/**/vectors/correct_train_*.json', recursive=True)
resume_roots = sorted({os.path.dirname(os.path.dirname(path)) for path in resume_markers})
assert len(resume_roots) <= 1, ('Attach at most one prior vector output', resume_roots)
if not os.path.exists(OUTPUT) and resume_roots and not FORCE_FRESH:
    print({'resume_from': resume_roots[0]})
    shutil.copytree(resume_roots[0], OUTPUT)
print({'dataset_root': dataset_root, 'checkpoint': checkpoint, 'output_exists': os.path.exists(OUTPUT)})

In [ ]:
def extraction_command(output, batch_size, chunk_size, smoke_limit=None):
    command = [
        sys.executable, os.path.join(WORKTREE, 'evaluation', 'extract_frozen_glim_vectors.py'),
        '--dataset-root', dataset_root, '--output-root', output, '--glim-root', GLIM_WORKTREE,
        '--checkpoint', checkpoint, '--glim-commit', GLIM_COMMIT,
        '--expected-index-sha256', EXPECTED_INDEX_SHA256,
        '--expected-checkpoint-sha256', CHECKPOINT_SHA256,
        '--expected-donor-sha256', DONOR_SHA256, '--device', 'cuda',
        '--batch-size', str(batch_size), '--chunk-size', str(chunk_size),
    ]
    if smoke_limit is not None:
        command += ['--smoke-limit', str(smoke_limit)]
    return command
if os.path.exists(SMOKE_OUTPUT):
    shutil.rmtree(SMOKE_OUTPUT)
subprocess.run(extraction_command(SMOKE_OUTPUT, 1, 1, smoke_limit=1), check=True)
smoke = json.load(open(os.path.join(SMOKE_OUTPUT, 'vector_manifest.json'), encoding='utf-8'))
assert smoke['status'] == 'pass' and smoke['run_mode'] == 'smoke'
assert set(smoke['condition_counts'].values()) == {1}
assert smoke['checks']['held_out_test_accessed'] is False
print({'smoke': 'PASS', 'conditions': smoke['condition_counts'], 'vector_dim': smoke['vector_dim']})

In [ ]:
# This is the long cell. Re-run it after interruption; completed hash-valid chunks are reused.
subprocess.run(extraction_command(OUTPUT, BATCH_SIZE, CHUNK_SIZE), check=True)

In [ ]:
manifest_path = os.path.join(OUTPUT, 'vector_manifest.json')
manifest = json.load(open(manifest_path, encoding='utf-8'))
expected_counts = {'correct_train': 17908, 'correct_val': 2200, 'matched_wrong_val': 2200, 'zero_val': 2200, 'gaussian_val': 2200}
assert manifest['status'] == 'pass' and manifest['run_mode'] == 'full_development'
assert manifest['condition_counts'] == expected_counts
assert manifest['source_index_sha256'] == EXPECTED_INDEX_SHA256
assert manifest['checkpoint_sha256'] == CHECKPOINT_SHA256
assert manifest['wrong_eeg_donor_sha256'] == DONOR_SHA256
assert manifest['glim_commit'] == GLIM_COMMIT and manifest['vector_dim'] == 1024
assert manifest['batch_size'] == BATCH_SIZE and manifest['chunk_size'] == CHUNK_SIZE
expected_checks = {
    'held_out_test_accessed': False,
    'target_identity_preserved': True,
    'matched_wrong_changes_signal_only': True,
    'zero_and_gaussian_keep_target_metadata': True,
    'gaussian_uses_training_statistics_only': True,
    'checkpoint_and_source_pinned': True,
    'vectors_finite': True,
}
assert manifest['checks'] == expected_checks, manifest['checks']
for chunk in manifest['chunks']:
    path = os.path.join(OUTPUT, 'vectors', f"{chunk['condition']}_{chunk['chunk_number']:05d}.npz")
    state = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            state.update(block)
    assert state.hexdigest() == chunk['vector_npz_sha256'], path
index_path = os.path.join(OUTPUT, 'vector_index.csv')
state = hashlib.sha256()
with open(index_path, 'rb') as handle:
    for block in iter(lambda: handle.read(1024 * 1024), b''):
        state.update(block)
assert state.hexdigest() == manifest['vector_index_sha256']
with open(index_path, encoding='utf-8', newline='') as handle:
    vector_index = list(csv.DictReader(handle))
assert len(vector_index) == 26708 and all(row['phase'] != 'test' for row in vector_index)
run_metadata = {
    'status': 'pass', 'project_commit': COMMIT, 'glim_commit': GLIM_COMMIT,
    'checkpoint_sha256': CHECKPOINT_SHA256, 'dataset_index_sha256': EXPECTED_INDEX_SHA256,
    'python': platform.python_version(), 'torch': torch.__version__, 'cuda': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0), 'batch_size': BATCH_SIZE, 'chunk_size': CHUNK_SIZE,
}
with open(os.path.join(OUTPUT, 'run_metadata.json'), 'w', encoding='utf-8') as handle:
    json.dump(run_metadata, handle, indent=2, sort_keys=True)
    handle.write('\n')
for path in (WORKTREE, GLIM_WORKTREE, SMOKE_OUTPUT):
    if os.path.exists(path):
        shutil.rmtree(path)
print({'rows': len(vector_index), 'chunks': len(manifest['chunks']), 'condition_counts': manifest['condition_counts'], 'vector_index_sha256': manifest['vector_index_sha256']})
print('FROZEN GLIM VECTOR EXTRACTION: PASS')